# KubeVision on FABRIC

Runbook for deploying KubeVision (CSC 478 project) on a 3-node RKE2 cluster, built with the same
Ansible playbooks as the course's [Resource Observation](https://linhbngo.github.io/csc478/09-resource-observation/) notebook.

| Role | Node | Count | Resources |
|------|------|-------|-----------|
| Control plane (`rke2-server`) | `node1` (`192.168.1.1`) | 1 | 4 cores, 8 GB RAM, 40 GB disk |
| Worker (`rke2-agent`) | `node2`, `node3` | 2 | 4 cores, 8 GB RAM, 40 GB disk |

Cluster traffic uses a private L2 dataplane (`192.168.1.0/24`). Ansible reaches the nodes over the FABRIC management network.

**Steps:** create the slice → install RKE2 (plus an NFS share for Milestone 3) → check that the nodes can reach `ghcr.io` → deploy
KubeVision → smoke test → open the UI from your laptop → screenshots → capture the Milestone 2 evidence → delete the slice.

**Before you start**

- The environment is set up as in the repository `Readme.md` (`fabric_rc`, token, keys), and
  `docker-compose.yml` mounts this folder: `- ./project:/home/fabric/project`.
- The five `ghcr.io/angeudybp/kubevision-*:0.2.0` images are pushed and **public**.

In [ ]:
nodeCnt = 3    # node1 is the RKE2 server, the rest are agents
coreCnt = 4    # CPUs per node
memSize = 8    # GB of memory per node
diskSize = 40  # GB of disk per node (each node pulls ~1.4 GB of KubeVision images)

REPO_URL = "https://github.com/angeudybp/csc478-project.git"
REPO_DIR = "kubevision"   # checkout on node1
TUNNEL_PORT = 3000        # published by docker-compose.yml; the UI opens at http://localhost:3000

## Import the FABlib Library

In [ ]:
from pathlib import Path
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager
import sys

for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "utils" / "ansible.py").exists():
        sys.path.insert(0, str(_candidate))
        break
else:
    raise ImportError("Could not find repo root containing utils/ansible.py")

from utils.ansible import DEFAULT_SSH_CONFIG, generate_rke2_playbooks

fablib = fablib_manager()
fablib.verify_and_configure()
fablib.create_ssh_tunnel_config(overwrite=False)

In [ ]:
slicePrefix = "angeudy-kubevision-"
network_name = "kvnet"
nic_name = "kvnic"

# Ask for twice the resources needed, so the slice lands on a site with headroom.
minCores = nodeCnt * coreCnt * 2
minMem = nodeCnt * memSize * 2

## 1. Create the slice

Tries each site with enough free capacity until a slice comes up. A slice left over from an earlier
run of this notebook, at the same site, is deleted first.

In [ ]:
from ipaddress import IPv4Network

resources = fablib.get_resources()
resources.update()
sites = resources.get_site_names()
selectedSite = ""

for site in sites:
    cores = resources.get_core_available(site)
    ram = resources.get_ram_available(site)
    if cores < minCores or ram < minMem:
        continue
    sliceName = slicePrefix + site

    # Clean up left-over slices with the same names
    for s in fablib.get_slices():
        if s.get_name() == sliceName:
            print(f"Deleting existing slice: {sliceName}")
            s.delete()
            break

    # Set up new slice
    slice = fablib.new_slice(name=sliceName)
    net = slice.add_l2network(name=network_name, subnet=IPv4Network("192.168.1.0/24"))

    for i in range(1, nodeCnt + 1):
        node = slice.add_node(name=f"node{i}",
                              site=site,
                              cores=coreCnt, ram=memSize, disk=diskSize,
                              image="default_ubuntu_22")
        iface = node.add_component(model="NIC_Basic", name=nic_name).get_interfaces()[0]
        iface.set_mode("config")
        net.add_interface(iface)

    print("==========================================================================")
    try:
        print(f"Submitting slice at {site}...")
        slice.submit(progress=False, wait=True)
    except Exception as e:
        print(f"{site}: slice submission failed with error {e}")
        try:
            slice.delete()
        except Exception:
            pass
        continue

    # Setup networking
    for i in range(1, nodeCnt + 1):
        node = slice.get_node(name=f"node{i}")
        iface = node.get_interface(network_name=network_name)
        iface.ip_link_up()
        iface.ip_addr_add(addr=f"192.168.1.{i}", subnet=IPv4Network("192.168.1.0/24"))
        print(f"{node.get_name()} -> 192.168.1.{i}")

    slice.update()
    if slice.get_state() == "Closing":
        print("Need to find new site")
        continue
    selectedSite = site
    print(f"Slice at {selectedSite} is stable: {slice.isStable()}")
    break

if not selectedSite:
    raise RuntimeError("No site could host the slice; try again later")

## 2. Install RKE2

Generates the Ansible inventory and playbooks from `templates/` (copied from the course), then runs
the course's three playbooks: host prerequisites, NFS (node1 exports `/opt/data` to the other nodes;
Milestone 3's persistent volume will live there), and RKE2. Output goes to `logs/`; a failed playbook prints the end of its log.

The cluster join token is random and kept in `playbook/rke2-token` (git-ignored), so rerunning this
cell against the same cluster reuses it.

In [ ]:
import secrets
import subprocess

TEMPLATES_DIR = Path("templates").resolve()
PLAYBOOK_DIR = Path("playbook").resolve()
LOGS_DIR = Path("logs").resolve()
PLAYBOOK_DIR.mkdir(exist_ok=True)
LOGS_DIR.mkdir(exist_ok=True)
assert (TEMPLATES_DIR / "host.yml.template").exists(), f"not found in {TEMPLATES_DIR}"

token_file = PLAYBOOK_DIR / "rke2-token"
if not token_file.exists():
    token_file.write_text(secrets.token_urlsafe(32))
    token_file.chmod(0o600)

sliceName = slicePrefix + selectedSite
slice = fablib.get_slice(name=sliceName)
generate_rke2_playbooks(slice, fablib, network_name,
                        templates_dir=TEMPLATES_DIR, playbook_dir=PLAYBOOK_DIR,
                        rke2_token=token_file.read_text().strip())
(PLAYBOOK_DIR / "inventory.yml").chmod(0o600)  # it contains the token


def run_playbook(name):
    log_path = LOGS_DIR / f"{selectedSite}-ansible-{name}.log"
    print(f"==== Running {name} playbook (log: {log_path.name}) ====")
    with open(log_path, "w") as log_file:
        result = subprocess.run(["ansible-playbook", "-i", PLAYBOOK_DIR / "inventory.yml",
                                 PLAYBOOK_DIR / f"playbook-{name}.yml"],
                                stdout=log_file, stderr=subprocess.STDOUT, text=True)
    if result.returncode != 0:
        print("".join(log_path.read_text().splitlines(keepends=True)[-30:]))
        raise RuntimeError(f"{selectedSite}: {name} playbook failed")
    print(f"{name} playbook is completed")


run_playbook("prereqs")
run_playbook("nfs")
run_playbook("rke2")

The RKE2 playbook ends by deploying an `nginx-demo` on NodePort 30080 to prove the cluster works.
KubeVision's gateway needs that port, so the demo is removed here after the check.

In [ ]:
server = slice.get_node("node1")
SERVER_IP = "192.168.1.1"


def run(node, command):
    """Run a command on a node and print what it printed."""
    stdout, stderr = node.execute(command, quiet=True)
    print(stdout + stderr)
    return stdout


print("==== nodes ====")
run(server, "kubectl get nodes -o wide")

print("==== nginx-demo via NodePort on the dataplane ====")
status = run(server, f"curl -s -o /dev/null -w '%{{http_code}}' http://{SERVER_IP}:30080/")
print("RKE2 works" if status.strip() == "200" else "nginx-demo did not answer 200; check logs/")
run(server, "kubectl delete deployment,svc nginx-demo --ignore-not-found")

## 3. Check that every node can reach the image registry

Each node pulls KubeVision's images from `ghcr.io`. **HTTP 401** means the registry is reachable
(401 only means "not logged in", which is fine for public images). **000** means that node cannot
reach it. In that case, build the images on your laptop and copy them in with
`fabric/load-images.sh` from the KubeVision repository, or delete the slice and use another site.

In [ ]:
for node in slice.get_nodes():
    out, _ = node.execute("curl -s -o /dev/null -m 15 -w '%{http_code}' https://ghcr.io/v2/", quiet=True)
    print(f"{node.get_name():6} {node.get_management_ip():40} ghcr.io -> HTTP {out.strip()}")

## 4. Deploy KubeVision

Clones the repository on `node1` and applies the manifests. The first start takes a few minutes while
every node pulls the images. The inference pods should be spread over more than one node (NODE column).

In [ ]:
run(server, f"[ -d {REPO_DIR} ] && git -C {REPO_DIR} pull --ff-only || git clone {REPO_URL} {REPO_DIR}")
run(server, f"cd {REPO_DIR} && kubectl apply -f deployments/ -f services/")
run(server, "for d in gateway preprocess inference storage; do "
            "kubectl -n imgpipe rollout status deploy/$d --timeout=600s; done")
run(server, "kubectl -n imgpipe get deploy,pods,svc -o wide")

Smoke test from `node1`, through the gateway NodePort. It should end with `18 passed, 0 failed`.

In [ ]:
run(server, f"cd {REPO_DIR} && GATEWAY_URL=http://{SERVER_IP}:30080 scripts/smoke.sh")

## 5. Open the UI from your laptop

Starts an SSH tunnel from this container, through the FABRIC bastion, to the gateway NodePort.
The container publishes port 3000, so on your laptop:

- open **http://localhost:3000** in a browser, or
- from the KubeVision repository: `GATEWAY_URL=http://localhost:3000 scripts/smoke.sh`

The tunnel lives as long as this notebook's kernel. Port 3000 is published on all of your machine's
network interfaces, so other devices on your network can reach the UI while it runs.

In [ ]:
import time
import urllib.request

slice_key = fablib.get_default_slice_key()["slice_private_key_file"]

tunnel = subprocess.Popen([
    "ssh", "-F", DEFAULT_SSH_CONFIG, "-i", slice_key, "-N",
    "-o", "ExitOnForwardFailure=yes", "-o", "ServerAliveInterval=30",
    "-o", "StrictHostKeyChecking=accept-new",
    "-L", f"0.0.0.0:{TUNNEL_PORT}:{SERVER_IP}:30080",
    f"{server.get_username()}@{server.get_management_ip()}",
])
time.sleep(5)
if tunnel.poll() is not None:
    raise RuntimeError(f"tunnel exited with code {tunnel.returncode}; is port {TUNNEL_PORT} already in use?")
with urllib.request.urlopen(f"http://localhost:{TUNNEL_PORT}/healthz", timeout=10) as r:
    print(f"Tunnel is up (gateway /healthz -> HTTP {r.status}). Open http://localhost:{TUNNEL_PORT}")

In [ ]:
# Run to close the tunnel.
# tunnel.terminate()

## 6. Screenshots for the README

Take these **before** step 7, which deletes and recreates the stack.
Open a **Terminal** in JupyterLab, paste the command printed below to log in to `node1`, and capture:

1. `kubectl -n imgpipe get deploy,pods,svc -o wide` (`01-get.png`)
2. `kubectl -n imgpipe scale deploy/inference --replicas=5`, then
   `kubectl -n imgpipe get pods -o wide` (`02-scale.png`), then scale back to `--replicas=3`
3. `kubectl -n imgpipe delete pod <one inference pod> --wait=false`, then
   `kubectl -n imgpipe get pods -o wide -w` showing its replacement, Ctrl-C once it is `2/2` (`03-self-healing.png`)
4. The UI at http://localhost:3000 with a classified image (`04-ui.png`)

Save them in `docs/evidence/m2/screenshots/` in the repository.

In [ ]:
print(server.get_ssh_command())

## 7. Capture the Milestone 2 evidence

Runs `scripts/evidence.sh fabric` on `node1` (about 5 minutes): overview, scaling 1 → 3, deleting an
inference pod while uploads continue, scale-to-zero, load balancing, and a full delete and re-apply.
The UI tunnel keeps working afterwards, because the NodePort stays 30080.

The output is unpacked into `evidence/fabric/` next to this notebook. Copy that folder to
`docs/evidence/m2/fabric/` in the KubeVision repository and commit it.

In [ ]:
import tarfile

stdout, _ = server.execute(f"cd {REPO_DIR} && GATEWAY_URL=http://{SERVER_IP}:30080 scripts/evidence.sh fabric "
                           "> evidence-run.log 2>&1; echo exit=$?; tail -5 evidence-run.log", quiet=True)
print(stdout)

server.execute(f"tar czf evidence-fabric.tgz -C {REPO_DIR}/docs/evidence/m2 fabric", quiet=True)
Path("evidence").mkdir(exist_ok=True)
server.download_file(str(Path("evidence/evidence-fabric.tgz").resolve()), "evidence-fabric.tgz")
with tarfile.open("evidence/evidence-fabric.tgz") as tar:
    tar.extractall("evidence", filter="data")
print(sorted(p.name for p in Path("evidence/fabric").iterdir()))

## 8. Delete the slice

Slices expire on their own, but deleting frees the resources for others. Everything needed to rebuild
is in this notebook. Close the tunnel first.

In [ ]:
# tunnel.terminate()
# slice.delete()